In [1]:
from nba_api.stats.endpoints import leaguedashplayerstats
from nba_api.stats.static import players
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import time
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")
print("Today we build the Player DNA Similarity Engine")
print("Find the 5 most statistically similar players to anyone in NBA history")

All imports successful!
Today we build the Player DNA Similarity Engine
Find the 5 most statistically similar players to anyone in NBA history


In [3]:
print("Fetching all NBA player stats for 2024-25 season...")
print("This might take 10-15 seconds...")

time.sleep(1)

player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2024-25',
    per_mode_detailed='PerGame'
)

df = player_stats.get_data_frames()[0]

print(f"\nPlayers fetched: {len(df)}")
print(f"Stats available: {len(df.columns)}")
print(f"\nSample columns: {list(df.columns[:15])}")

Fetching all NBA player stats for 2024-25 season...
This might take 10-15 seconds...

Players fetched: 569
Stats available: 67

Sample columns: ['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M']


In [4]:
# Select the stats that best define a player's style
# We want a mix of scoring, playmaking, defense and efficiency
# These 15 stats capture how a player plays, not just how much they score

similarity_features = [
    'PTS',      # Scoring volume
    'AST',      # Playmaking
    'REB',      # Rebounding
    'STL',      # Defensive activity
    'BLK',      # Rim protection
    'TOV',      # Ball security
    'FG_PCT',   # Shooting efficiency
    'FG3M',     # Three point volume
    'FG3_PCT',  # Three point accuracy
    'FT_PCT',   # Free throw shooting
    'MIN',      # Usage and role
    'PLUS_MINUS' # Overall impact
]

# Filter to players with meaningful minutes
# A player with 2 minutes per game is not comparable to a starter
# 15+ minutes per game gives us legitimate rotation players and starters
df_filtered = df[df['MIN'] >= 15].copy()
df_filtered = df_filtered.dropna(subset=similarity_features)

print(f"Players with 15+ minutes per game: {len(df_filtered)}")
print(f"\nFeatures we will use for similarity:")
for i, feat in enumerate(similarity_features, 1):
    print(f"  {i:2}. {feat}")

# Show the top 10 scorers to verify our data looks right
print(f"\nTop 10 scorers this season:")
top_scorers = df_filtered.nlargest(10, 'PTS')[['PLAYER_NAME', 'PTS', 'AST', 'REB', 'MIN']]
print(top_scorers.to_string())

Players with 15+ minutes per game: 370

Features we will use for similarity:
   1. PTS
   2. AST
   3. REB
   4. STL
   5. BLK
   6. TOV
   7. FG_PCT
   8. FG3M
   9. FG3_PCT
  10. FT_PCT
  11. MIN
  12. PLUS_MINUS

Top 10 scorers this season:
                 PLAYER_NAME   PTS   AST   REB   MIN
490  Shai Gilgeous-Alexander  32.7   6.4   5.0  34.2
180    Giannis Antetokounmpo  30.4   6.5  11.9  34.2
423             Nikola Jokić  29.6  10.2  12.7  36.7
366              Luka Dončić  28.2   7.7   8.2  35.4
29           Anthony Edwards  27.6   4.5   5.7  36.3
263             Jayson Tatum  26.8   6.0   8.7  36.4
330             Kevin Durant  26.6   4.2   6.0  36.5
545             Tyrese Maxey  26.3   6.1   3.3  37.7
69           Cade Cunningham  26.1   9.1   6.1  35.0
226            Jalen Brunson  26.0   7.3   2.9  35.4


In [5]:
# Step 1: Normalize the data using StandardScaler
# Remember our analogy: we need all stats on the same scale
# so points (0-35) and FG% (0.40-0.65) are treated equally

scaler = StandardScaler()

# Extract just the feature columns as a numpy array
X = df_filtered[similarity_features].values

# Fit and transform: calculate mean and std, then normalize
X_scaled = scaler.fit_transform(X)

print("Data normalized successfully!")
print(f"Original PTS range: {df_filtered['PTS'].min():.1f} to {df_filtered['PTS'].max():.1f}")
print(f"Scaled PTS range:   {X_scaled[:, 0].min():.2f} to {X_scaled[:, 0].max():.2f}")
print("\nNow every stat is on the same scale.")
print("The algorithm treats scoring and shooting efficiency equally.")

# Step 2: Calculate cosine similarity between all players
# This creates a 370x370 matrix where each cell is the similarity
# between player i and player j
# 1.0 = identical statistical profile
# 0.0 = completely different
similarity_matrix = cosine_similarity(X_scaled)

print(f"\nSimilarity matrix shape: {similarity_matrix.shape}")
print("Each cell contains a similarity score between 0 and 1")
print("1.0 = identical player, 0.0 = completely different player")

Data normalized successfully!
Original PTS range: 3.0 to 32.7
Scaled PTS range:   -1.48 to 3.41

Now every stat is on the same scale.
The algorithm treats scoring and shooting efficiency equally.

Similarity matrix shape: (370, 370)
Each cell contains a similarity score between 0 and 1
1.0 = identical player, 0.0 = completely different player


In [6]:
# Step 3: Build the similarity search function
# This is the core of our Player DNA engine

def find_similar_players(player_name: str, top_n: int = 5) -> list:
    """
    Find the most statistically similar players to a given player.
    
    How it works:
    1. Find the player's row in our dataset
    2. Look up their similarity scores against all other players
    3. Sort by similarity score descending
    4. Return the top N most similar players
    """
    
    # Find the player in our dataset
    # str.contains is case insensitive search
    mask = df_filtered['PLAYER_NAME'].str.contains(player_name, case=False, na=False)
    matches = df_filtered[mask]
    
    if len(matches) == 0:
        return {"error": f"Player '{player_name}' not found in current season data"}
    
    # Get the index of this player in our filtered dataframe
    player_idx = matches.index[0]
    
    # Convert to positional index for the similarity matrix
    filtered_positions = list(df_filtered.index)
    pos = filtered_positions.index(player_idx)
    
    # Get similarity scores for this player against everyone
    similarity_scores = similarity_matrix[pos]
    
    # Get indices sorted by similarity score descending
    # argsort returns indices that would sort the array
    # [::-1] reverses it so highest similarity comes first
    sorted_indices = np.argsort(similarity_scores)[::-1]
    
    # Skip index 0 because that is the player themselves (similarity = 1.0)
    similar_players = []
    for idx in sorted_indices[1:top_n+1]:
        similar_player = df_filtered.iloc[idx]
        similar_players.append({
            "name": similar_player['PLAYER_NAME'],
            "team": similar_player['TEAM_ABBREVIATION'],
            "similarity_score": round(float(similarity_scores[idx]), 3),
            "pts": similar_player['PTS'],
            "ast": similar_player['AST'],
            "reb": similar_player['REB'],
            "fg_pct": round(similar_player['FG_PCT'] * 100, 1)
        })
    
    # Get the target player's stats too
    target = df_filtered.iloc[pos]
    
    return {
        "player": player_name,
        "stats": {
            "pts": target['PTS'],
            "ast": target['AST'],
            "reb": target['REB'],
            "fg_pct": round(target['FG_PCT'] * 100, 1)
        },
        "similar_players": similar_players
    }

print("Player DNA engine built successfully!")
print("Testing with LeBron James...")

result = find_similar_players("LeBron James")
print(f"\nPlayer: {result['player']}")
print(f"Stats: {result['stats']}")
print(f"\nMost similar players:")
for i, p in enumerate(result['similar_players'], 1):
    print(f"  {i}. {p['name']} ({p['team']}) - Similarity: {p['similarity_score']}")
    print(f"     {p['pts']} PPG, {p['ast']} APG, {p['reb']} RPG, {p['fg_pct']}% FG")

Player DNA engine built successfully!
Testing with LeBron James...

Player: LeBron James
Stats: {'pts': np.float64(24.4), 'ast': np.float64(8.2), 'reb': np.float64(7.8), 'fg_pct': np.float64(51.3)}

Most similar players:
  1. Cade Cunningham (DET) - Similarity: 0.958
     26.1 PPG, 9.1 APG, 6.1 RPG, 46.9% FG
  2. Josh Giddey (CHI) - Similarity: 0.908
     14.6 PPG, 7.2 APG, 8.1 RPG, 46.5% FG
  3. Deni Avdija (POR) - Similarity: 0.908
     16.9 PPG, 3.9 APG, 7.3 RPG, 47.6% FG
  4. Paolo Banchero (ORL) - Similarity: 0.898
     25.9 PPG, 4.8 APG, 7.5 RPG, 45.2% FG
  5. James Harden (LAC) - Similarity: 0.879
     22.8 PPG, 8.7 APG, 5.8 RPG, 41.0% FG


In [7]:
# Test with different player archetypes
# This proves our engine works across all playing styles

test_players = ["Nikola Jokic", "Steph Curry", "Anthony Davis"]

for player in test_players:
    print(f"\n{'='*50}")
    result = find_similar_players(player)
    
    if "error" in result:
        print(result["error"])
        continue
        
    print(f"Player DNA: {result['player']}")
    print(f"Stats: {result['stats']['pts']} PPG, {result['stats']['ast']} APG, {result['stats']['reb']} RPG")
    print(f"\nMost similar players:")
    for i, p in enumerate(result['similar_players'], 1):
        print(f"  {i}. {p['name']} ({p['team']}) - {p['similarity_score']} similarity")


Player 'Nikola Jokic' not found in current season data

Player 'Steph Curry' not found in current season data

Player DNA: Anthony Davis
Stats: 24.7 PPG, 3.5 APG, 11.6 RPG

Most similar players:
  1. Victor Wembanyama (SAS) - 0.869 similarity
  2. Evan Mobley (CLE) - 0.847 similarity
  3. P.J. Washington (DAL) - 0.824 similarity
  4. Amen Thompson (HOU) - 0.82 similarity
  5. Jalen Johnson (ATL) - 0.796 similarity


In [8]:
# The issue is special characters and nicknames
# Let's check the exact names in our dataset

print("Searching for Jokic:")
jokic_search = df_filtered[df_filtered['PLAYER_NAME'].str.contains('Joki', case=False, na=False)]
print(jokic_search[['PLAYER_NAME', 'TEAM_ABBREVIATION', 'PTS']].to_string())

print("\nSearching for Curry:")
curry_search = df_filtered[df_filtered['PLAYER_NAME'].str.contains('Curry', case=False, na=False)]
print(curry_search[['PLAYER_NAME', 'TEAM_ABBREVIATION', 'PTS']].to_string())

print("\nSearching for Luka:")
luka_search = df_filtered[df_filtered['PLAYER_NAME'].str.contains('Don', case=False, na=False)]
print(luka_search[['PLAYER_NAME', 'TEAM_ABBREVIATION', 'PTS']].to_string())

Searching for Jokic:
      PLAYER_NAME TEAM_ABBREVIATION   PTS
423  Nikola Jokić               DEN  29.6

Searching for Curry:
       PLAYER_NAME TEAM_ABBREVIATION   PTS
488     Seth Curry               CHA   6.5
498  Stephen Curry               GSW  24.5

Searching for Luka:
          PLAYER_NAME TEAM_ABBREVIATION   PTS
3        Aaron Gordon               DEN  14.7
54     Brandon Boston               NOP  10.7
55     Brandon Clarke               MEM   8.3
56     Brandon Ingram               TOR  22.2
57     Brandon Miller               CHA  21.0
148   Donovan Clingan               POR   6.5
149  Donovan Mitchell               CLE  24.0
150  Donte DiVincenzo               MIN  11.7
168       Eric Gordon               PHI   6.8
321    Keldon Johnson               SAS  12.7
366       Luka Dončić               LAL  28.2
374   Malcolm Brogdon               WAS  12.7
489    Shaedon Sharpe               POR  18.5
527   Trendon Watford               BKN  10.2


In [9]:
import unicodedata

def normalize_name(name: str) -> str:
    """
    Remove accents and special characters from names.
    Nikola Jokić -> Nikola Jokic
    Luka Dončić -> Luka Doncic
    """
    return ''.join(
        c for c in unicodedata.normalize('NFD', name)
        if unicodedata.category(c) != 'Mn'
    ).lower()

# Create a normalized name column for searching
df_filtered['PLAYER_NAME_NORMALIZED'] = df_filtered['PLAYER_NAME'].apply(normalize_name)

def find_similar_players(player_name: str, top_n: int = 5) -> dict:
    """
    Find the most statistically similar players.
    Now handles accents and special characters.
    """
    search_term = normalize_name(player_name)

    mask = df_filtered['PLAYER_NAME_NORMALIZED'].str.contains(
        search_term, case=False, na=False
    )
    matches = df_filtered[mask]

    if len(matches) == 0:
        return {"error": f"Player '{player_name}' not found"}

    player_idx = matches.index[0]
    filtered_positions = list(df_filtered.index)
    pos = filtered_positions.index(player_idx)

    similarity_scores = similarity_matrix[pos]
    sorted_indices = np.argsort(similarity_scores)[::-1]

    similar_players = []
    for idx in sorted_indices[1:top_n+1]:
        similar_player = df_filtered.iloc[idx]
        similar_players.append({
            "name": similar_player['PLAYER_NAME'],
            "team": similar_player['TEAM_ABBREVIATION'],
            "similarity_score": round(float(similarity_scores[idx]), 3),
            "pts": round(float(similar_player['PTS']), 1),
            "ast": round(float(similar_player['AST']), 1),
            "reb": round(float(similar_player['REB']), 1),
            "fg_pct": round(float(similar_player['FG_PCT']) * 100, 1)
        })

    target = df_filtered.iloc[pos]

    return {
        "player": target['PLAYER_NAME'],
        "stats": {
            "pts": round(float(target['PTS']), 1),
            "ast": round(float(target['AST']), 1),
            "reb": round(float(target['REB']), 1),
            "fg_pct": round(float(target['FG_PCT']) * 100, 1)
        },
        "similar_players": similar_players
    }

# Test all our players now
test_players = ["LeBron James", "Nikola Jokic", "Stephen Curry", "Luka Doncic", "Anthony Davis"]

for player in test_players:
    result = find_similar_players(player)
    if "error" in result:
        print(f"Not found: {player}")
        continue
    print(f"\n{result['player']} most similar to:")
    for i, p in enumerate(result['similar_players'], 1):
        print(f"  {i}. {p['name']} ({p['similarity_score']})")


LeBron James most similar to:
  1. Cade Cunningham (0.958)
  2. Josh Giddey (0.908)
  3. Deni Avdija (0.908)
  4. Paolo Banchero (0.898)
  5. James Harden (0.879)

Nikola Jokić most similar to:
  1. Luka Dončić (0.896)
  2. Josh Hart (0.874)
  3. Jalen Johnson (0.869)
  4. LeBron James (0.863)
  5. Jalen Williams (0.859)

Stephen Curry most similar to:
  1. Anthony Edwards (0.946)
  2. Tyler Herro (0.945)
  3. Damian Lillard (0.942)
  4. Austin Reaves (0.934)
  5. Jalen Green (0.913)

Luka Dončić most similar to:
  1. Jayson Tatum (0.939)
  2. James Harden (0.929)
  3. Desmond Bane (0.913)
  4. Jaylen Brown (0.905)
  5. Nikola Jokić (0.896)

Anthony Davis most similar to:
  1. Victor Wembanyama (0.869)
  2. Evan Mobley (0.847)
  3. P.J. Washington (0.824)
  4. Amen Thompson (0.82)
  5. Jalen Johnson (0.796)
